# General Creators: Kafka and PySpark ETL

Publish raw metadata to Kafka, parse streams in PySpark, clean video-level records, and build creator-month panel features.


##2. Kafka Producer

In [ ]:
# DO NOT RUN
import json
import pandas as pd
from kafka import KafkaProducer

KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "youtube_video_raw"

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

raw_df = pd.read_csv(f"{OUTPUT_DIR}/youtube_video_level_raw.csv")

for _, row in raw_df.iterrows():
    producer.send(KAFKA_TOPIC, row.to_dict())

producer.flush()
producer.close()

print("Finished sending raw video-level data to Kafka.")

Finished sending raw video-level data to Kafka.


##3. PySpark ETL

In [23]:
import pyspark

print("PySpark version:", pyspark.__version__)

PySpark version: 4.0.2


In [24]:
import json
import pandas as pd
from kafka import KafkaProducer

KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "youtube_video_raw"

raw_df = pd.read_csv(f"{OUTPUT_DIR}/youtube_video_level_raw.csv")

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v, default=str).encode("utf-8")
)

for _, row in raw_df.iterrows():
    record = row.where(pd.notnull(row), None).to_dict()
    producer.send(KAFKA_TOPIC, value=record)

producer.flush()
producer.close()

print(f"Finished sending {len(raw_df):,} raw video-level records to Kafka topic: {KAFKA_TOPIC}")

Finished sending 82,664 raw video-level records to Kafka topic: youtube_video_raw


In [26]:
import pyspark
from pyspark.sql import SparkSession

print("PySpark version:", pyspark.__version__)

try:
    spark.stop()
except:
    pass

SPARK_VERSION = pyspark.__version__

spark = (
    SparkSession.builder
    .appName("YouTube-Kafka-ETL")
    .config(
        "spark.jars.packages",
        f"org.apache.spark:spark-sql-kafka-0-10_2.13:{SPARK_VERSION}"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Kafka package loaded.")

PySpark version: 4.0.2
Spark version: 4.0.2
Kafka package loaded.


In [27]:
!./kafka_2.13-3.1.0/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --list

youtube_video_raw


In [28]:
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "youtube_video_raw"

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

json_df = kafka_df.selectExpr("CAST(value AS STRING) AS json_value")

json_df.printSchema()

root
 |-- json_value: string (nullable = true)



In [29]:
from pyspark.sql import functions as F

sample_spark_df = spark.createDataFrame(raw_df)

video_schema = sample_spark_df.schema

parsed_df = (
    json_df
    .select(F.from_json(F.col("json_value"), video_schema).alias("data"))
    .select("data.*")
)

parsed_df.printSchema()

root
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- playlist_published_at: string (nullable = true)
 |-- actual_channel_id: string (nullable = true)
 |-- actual_channel_title: string (nullable = true)
 |-- title: string (nullable = true)
 |-- actual_published_at: string (nullable = true)
 |-- duration_iso8601: string (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- view_count: double (nullable = true)
 |-- like_count: double (nullable = true)
 |-- comment_count: double (nullable = true)
 |-- privacy_status: string (nullable = true)



In [30]:
query = (
    parsed_df.writeStream
    .format("memory")
    .queryName("youtube_video_stream")
    .outputMode("append")
    .start()
)

In [31]:
spark.sql("SELECT * FROM youtube_video_stream LIMIT 10").show(truncate=False)
spark.sql("SELECT COUNT(*) AS row_count FROM youtube_video_stream").show()
query.stop()

+----------+-------------+--------+---------------------+-----------------+--------------------+-----+-------------------+----------------+----------------+----------+----------+-------------+--------------+
|channel_id|channel_title|video_id|playlist_published_at|actual_channel_id|actual_channel_title|title|actual_published_at|duration_iso8601|duration_seconds|view_count|like_count|comment_count|privacy_status|
+----------+-------------+--------+---------------------+-----------------+--------------------+-----+-------------------+----------------+----------------+----------+----------+-------------+--------------+
+----------+-------------+--------+---------------------+-----------------+--------------------+-----+-------------------+----------------+----------------+----------+----------+-------------+--------------+

+---------+
|row_count|
+---------+
|        0|
+---------+



In [32]:
STREAM_OUTPUT_PATH = f"{OUTPUT_DIR}/spark_stream_output/youtube_video_parquet"
CHECKPOINT_PATH = f"{OUTPUT_DIR}/spark_stream_output/checkpoints/youtube_video_parquet"

query = (
    parsed_df.writeStream
    .format("parquet")
    .option("path", STREAM_OUTPUT_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .start()
)
query.stop()

In [34]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("YouTube-Kafka-ETL")
    .config(
        "spark.jars.packages",
       "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    )
    .getOrCreate()
)

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "youtube_video_raw")
    .option("startingOffsets", "earliest")
    .load()
)

json_df = kafka_df.selectExpr("CAST(value AS STRING) as json_value")

In [35]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("video_id",             StringType(),  True),
    StructField("actual_channel_id",    StringType(),  True),
    StructField("actual_channel_title", StringType(),  True),
    StructField("title",                StringType(),  True),
    StructField("actual_published_at",  StringType(),  True),
    StructField("duration_iso8601",     StringType(),  True),
    StructField("duration_seconds",     IntegerType(), True),
    StructField("view_count",           DoubleType(),  True),
    StructField("like_count",           DoubleType(),  True),
    StructField("comment_count",        DoubleType(),  True),
    StructField("privacy_status",       StringType(),  True),
    StructField("upload_status",        StringType(),  True)
])

# For modeling, use the static API extract when available. The Kafka stream is
# retained as a pipeline component, but Spark streaming DataFrames cannot be
# converted directly to pandas or counted without writeStream sinks.
if "raw_df" in globals() and isinstance(raw_df, pd.DataFrame) and len(raw_df) > 0:
    parsed_df = spark.createDataFrame(raw_df.copy())
    for missing_col in ["privacy_status", "upload_status"]:
        if missing_col not in parsed_df.columns:
            parsed_df = parsed_df.withColumn(missing_col, F.lit(None).cast(StringType()))
    print(f"Using static raw_df extract for ETL/modeling: {len(raw_df):,} video rows")
else:
    parsed_df = json_df.select(
        F.from_json(F.col("json_value"), schema).alias("data")
    ).select("data.*")
    print("Using Kafka streaming source. For native Python modeling, write the stream to a static table or create raw_df first.")

parsed_df.printSchema()

Using static raw_df extract for ETL/modeling: 82,664 video rows
root
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- playlist_published_at: string (nullable = true)
 |-- actual_channel_id: string (nullable = true)
 |-- actual_channel_title: string (nullable = true)
 |-- title: string (nullable = true)
 |-- actual_published_at: string (nullable = true)
 |-- duration_iso8601: string (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- view_count: double (nullable = true)
 |-- like_count: double (nullable = true)
 |-- comment_count: double (nullable = true)
 |-- privacy_status: string (nullable = true)
 |-- upload_status: string (nullable = true)



In [36]:
POLICY_DATE       = "2010-12-01"
LONG_VIDEO_SECONDS = 15 * 60

clean_df = (
    parsed_df
    .filter(F.col("video_id").isNotNull())
    .filter(F.col("actual_channel_id").isNotNull())
    .withColumn("published_ts", F.to_timestamp("actual_published_at"))
    .filter(F.col("published_ts").isNotNull())
    .filter(F.col("privacy_status").isNull() | (F.col("privacy_status") == "public"))
    .dropDuplicates(["video_id"])
    .withColumn("month_date",  F.to_date(F.date_trunc("month", F.col("published_ts"))))
    .withColumn("year_month",  F.date_format(F.col("month_date"), "yyyy-MM"))
    .withColumn(
        "is_long_video",
        F.when(F.col("duration_seconds") > LONG_VIDEO_SECONDS, 1).otherwise(0)
    )
)

clean_df.printSchema()

root
 |-- channel_id: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- playlist_published_at: string (nullable = true)
 |-- actual_channel_id: string (nullable = true)
 |-- actual_channel_title: string (nullable = true)
 |-- title: string (nullable = true)
 |-- actual_published_at: string (nullable = true)
 |-- duration_iso8601: string (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- view_count: double (nullable = true)
 |-- like_count: double (nullable = true)
 |-- comment_count: double (nullable = true)
 |-- privacy_status: string (nullable = true)
 |-- upload_status: string (nullable = true)
 |-- published_ts: timestamp (nullable = true)
 |-- month_date: date (nullable = true)
 |-- year_month: string (nullable = true)
 |-- is_long_video: integer (nullable = false)



#4. Eligibility-Based Treatment Definition
The DID treatment below is based only on eligibility observed before the policy date. In this pipeline, the default proxy is whether a creator had already posted at least one video longer than 15 minutes before December 2010, which indicates that the account was already eligible/whitelisted for long uploads before the policy shock. If separate verification-status data are available later, merge it into channel_eligibility_df and use that indicator instead.

In [37]:
STUDY_START_MONTH = "2010-01-01"
STUDY_END_MONTH   = "2011-12-01"   # inclusive month start

# Treatment must be predetermined. This proxy uses only pre-policy behavior that
# reveals eligibility/whitelisting, not post-policy adoption of longer videos.
channel_eligibility_df = (
    clean_df
    .filter(F.col("published_ts") < F.lit(POLICY_DATE))
    .groupBy("actual_channel_id")
    .agg(
        F.count("video_id").alias("pre_policy_video_count"),
        F.sum("view_count").alias("pre_policy_views_sum"),
        F.sum(F.when(F.col("duration_seconds") > LONG_VIDEO_SECONDS, 1).otherwise(0)).alias("pre_policy_long_video_count")
    )
    .withColumn(
        "eligible_creator",
        F.when(F.col("pre_policy_long_video_count") > 0, 1).otherwise(0)
    )
)


channel_eligibility_df.groupBy("eligible_creator").count().show()


+----------------+-----+
|eligible_creator|count|
+----------------+-----+
|               1|   54|
|               0|  337|
+----------------+-----+



In [38]:
monthly_agg_df = (
    clean_df
    .groupBy("actual_channel_id", "month_date", "year_month")
    .agg(
        F.first("actual_channel_title", ignorenulls=True).alias("actual_channel_title"),
        F.count("video_id").alias("monthly_video_count"),
        F.sum("is_long_video").alias("monthly_long_video_count"),
        F.sum(F.coalesce(F.col("view_count"), F.lit(0))).alias("monthly_views_sum_current"),
        F.sum(F.coalesce(F.col("like_count"), F.lit(0))).alias("monthly_likes_sum_current"),
        F.sum(F.coalesce(F.col("comment_count"), F.lit(0))).alias("monthly_comments_sum_current"),
        F.avg("duration_seconds").alias("avg_duration_seconds")
    )
)

channels_spine_df = (
    clean_df
    .groupBy("actual_channel_id")
    .agg(F.first("actual_channel_title", ignorenulls=True).alias("actual_channel_title"))
)

months_spine_df = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{STUDY_START_MONTH}'),
        to_date('{STUDY_END_MONTH}'),
        interval 1 month
    )) AS month_date
""").withColumn("year_month", F.date_format(F.col("month_date"), "yyyy-MM"))

panel_df = (
    channels_spine_df
    .crossJoin(months_spine_df)
    .join(monthly_agg_df.drop("actual_channel_title", "year_month"),
          on=["actual_channel_id", "month_date"], how="left")
    .join(channel_eligibility_df, on="actual_channel_id", how="left")
    .fillna({
        "monthly_video_count": 0,
        "monthly_long_video_count": 0,
        "monthly_views_sum_current": 0,
        "monthly_likes_sum_current": 0,
        "monthly_comments_sum_current": 0,
        "pre_policy_video_count": 0,
        "pre_policy_views_sum": 0,
        "pre_policy_long_video_count": 0,
        "eligible_creator": 0,
    })
    .withColumn("post_policy", F.when(F.col("month_date") >= F.lit(POLICY_DATE), 1).otherwise(0))
    .withColumn("did_eligible", F.col("eligible_creator") * F.col("post_policy"))
    .withColumn("any_long_video_this_month", F.when(F.col("monthly_long_video_count") > 0, 1).otherwise(0))
    .withColumn(
        "event_time_policy_months",
        F.year("month_date") * 12 + F.month("month_date")
        - (F.year(F.to_date(F.lit(POLICY_DATE))) * 12 + F.month(F.to_date(F.lit(POLICY_DATE))))
    )
)

panel_df.cache()
print(f"Panel rows: {panel_df.count():,}")
panel_df.groupBy("eligible_creator", "post_policy").agg(
    F.count("*").alias("channel_months"),
    F.sum("monthly_video_count").alias("videos"),
    F.sum("monthly_views_sum_current").alias("views")
).orderBy("eligible_creator", "post_policy").show()


Panel rows: 11,856
+----------------+-----------+--------------+------+---------------+
|eligible_creator|post_policy|channel_months|videos|          views|
+----------------+-----------+--------------+------+---------------+
|               0|          0|          4840| 15733|6.8499770844E10|
|               0|          1|          5720| 41900|8.6883827613E10|
|               1|          0|           594| 11887|1.7860594069E10|
|               1|          1|           702| 13144|2.5774737806E10|
+----------------+-----------+--------------+------+---------------+

